In [4]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/tyre_data.csv')

#checking the shape of the current dataset
print(df.shape)
print(df.head())


Mounted at /content/drive
(172030, 34)
      timestamp  run_block  tyre_pressure_set  camber_set  track_temp_set  \
0  1.776520e+09          1               20.0        -2.5            25.0   
1  1.776520e+09          1               20.0        -2.5            25.0   
2  1.776520e+09          1               20.0        -2.5            25.0   
3  1.776520e+09          1               20.0        -2.5            25.0   
4  1.776520e+09          1               20.0        -2.5            25.0   

   lap_number  speed_kmh  throttle  brake  gear  ...  pressure_RL  \
0           0       1.56       0.0    0.0     1  ...        15.84   
1           0       0.99       0.0    1.0     1  ...        15.84   
2           0       0.32       0.0    1.0     1  ...        15.84   
3           0       0.71       0.0    1.0     1  ...        15.84   
4           0       0.67       0.0    1.0     1  ...        15.84   

   pressure_RR  camber_FL  camber_FR  camber_RL  camber_RR  temp_FL  temp_FR  \
0  

In [5]:
#removing the columns that i wont need
df = df.drop(columns=['timestamp', 'run_block', 'tyre_pressure_set', 'camber_set', 'track_temp_set', 'lap_number', 'gear'])
print(df.shape)
print(df.head())

(172030, 27)
   speed_kmh  throttle  brake   rpms  steer_angle  long_accel_g  lat_accel_g  \
0       1.56       0.0    0.0   69.4          0.0           0.0          0.0   
1       0.99       0.0    1.0  148.7          0.0           0.0          0.0   
2       0.32       0.0    1.0  204.9          0.0           0.0          0.0   
3       0.71       0.0    1.0  251.2          0.0           0.0          0.0   
4       0.67       0.0    1.0  281.0          0.0           0.0          0.0   

   slip_FL  slip_FR  slip_RL  ...  pressure_RL  pressure_RR  camber_FL  \
0   0.0000   0.6719   0.0837  ...        15.84        15.84    -0.0655   
1   0.1546   0.6806   0.1215  ...        15.84        15.84    -0.0556   
2   0.1813   0.7833   0.0363  ...        15.84        15.84    -0.0604   
3   0.2062   0.1797   0.0105  ...        15.84        15.84    -0.0658   
4   0.1130   0.0834   0.0744  ...        15.84        15.84    -0.0671   

   camber_FR  camber_RL  camber_RR  temp_FL  temp_FR  temp_RL

In [6]:
#output columns
output_cols = ['temp_FL', 'temp_FR', 'temp_RL', 'temp_RR']

#input features
input_cols = [col for col in df.columns if col not in output_cols]

X = df[input_cols]
y = df[output_cols]

print('Features:', input_cols)
print('Outputs:', output_cols)

Features: ['speed_kmh', 'throttle', 'brake', 'rpms', 'steer_angle', 'long_accel_g', 'lat_accel_g', 'slip_FL', 'slip_FR', 'slip_RL', 'slip_RR', 'load_FL', 'load_FR', 'load_RL', 'load_RR', 'pressure_FL', 'pressure_FR', 'pressure_RL', 'pressure_RR', 'camber_FL', 'camber_FR', 'camber_RL', 'camber_RR']
Outputs: ['temp_FL', 'temp_FR', 'temp_RL', 'temp_RR']


In [7]:
from sklearn.preprocessing import StandardScaler
import numpy as np


scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42) #80-20 split

print('Training set:', X_train.shape)
print('Test set:', X_test.shape)

Training set: (137624, 23)
Test set: (34406, 23)


In [9]:
#converting data into PyTorch tensors
import torch
import torch.nn as nn

X_train_t = torch.tensor(X_train, dtype=torch.float32) #each number is stored as a 32bit floating point number
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

In [12]:
#network architecture
class TyrePINN(nn.Module):
  def __init__(self, input_size, output_size):
    super(TyrePINN, self).__init__()
    self.network = nn.Sequential(
        nn.Linear(input_size, 64),
            nn.Tanh(),
            nn.Linear(64, 128),
            nn.Tanh(),
            nn.Linear(128, 128),
            nn.Tanh(),
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, output_size)
    )

  def forward(self, x):
    return self.network(x)

#initliasing the model
model = TyrePINN(input_size=23, output_size=4)
print('Total parameters:', sum(p.numel() for p in model.parameters()))

Total parameters: 34884


In [18]:
import torch.optim as optim

learning_rate = 1e-3
epochs = 1000 #number of iterations
lambda_physics = 0.1 #weight of the physics loss

optimiser = optim.Adam(model.parameters(), lr=learning_rate)
mse_loss = nn.MSELoss()

train_losses = []
val_losses = []
for epoch in range(epochs):
  model.train()
  optimiser.zero_grad()
  y_pred = model(X_train_t) #forward pass
  loss_data = mse_loss(y_pred, y_train_t)

  dT_pred = y_pred[1:] - y_pred[:-1] #the rate of temp change should relate to slip and load. Approximating dT/dt using consec differences
  slip_train = X_train_t[1:, 6:10] #slip columns for each tyre
  load_train = X_train_t[1:, 10:14] #load columns for each tyre
  Q_in = (slip_train * load_train)
  Q_out = y_pred[1:]
  physics_residual = dT_pred - (Q_in - 0.01 * Q_out)
  loss_physics = mse_loss(physics_residual, torch.zeros_like(physics_residual))

  loss = loss_data + lambda_physics * loss_physics #total loss including the physics weighting
  loss.backward()
  optimiser.step()

  model.eval() #validation losses
  with torch.no_grad():
      y_val_pred = model(X_test_t)
      val_loss = mse_loss(y_val_pred, y_test_t)

  train_losses.append(loss_data.item())
  val_losses.append(val_loss.item())

  if epoch % 100 == 0:
      print(f'Epoch {epoch} | Data Loss: {loss_data.item():.4f} | Physics Loss: {loss_physics.item():.4f} | Val Loss: {val_loss.item():.4f}')


KeyboardInterrupt: 